In [2]:
import pandas as pd
df = pd.read_csv("retail_store_sales.csv")
df.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [3]:
df.shape

(12575, 11)

In [4]:
df.columns

Index(['Transaction ID', 'Customer ID', 'Category', 'Item', 'Price Per Unit',
       'Quantity', 'Total Spent', 'Payment Method', 'Location',
       'Transaction Date', 'Discount Applied'],
      dtype='object')

In [5]:
df.info

<bound method DataFrame.info of       Transaction ID Customer ID       Category          Item  Price Per Unit  \
0        TXN_6867343     CUST_09     Patisserie   Item_10_PAT            18.5   
1        TXN_3731986     CUST_22  Milk Products  Item_17_MILK            29.0   
2        TXN_9303719     CUST_02       Butchers   Item_12_BUT            21.5   
3        TXN_9458126     CUST_06      Beverages   Item_16_BEV            27.5   
4        TXN_4575373     CUST_05           Food   Item_6_FOOD            12.5   
...              ...         ...            ...           ...             ...   
12570    TXN_9347481     CUST_18     Patisserie   Item_23_PAT            38.0   
12571    TXN_4009414     CUST_03      Beverages    Item_2_BEV             6.5   
12572    TXN_5306010     CUST_11       Butchers    Item_7_BUT            14.0   
12573    TXN_5167298     CUST_04      Furniture    Item_7_FUR            14.0   
12574    TXN_2407494     CUST_23           Food   Item_9_FOOD            17.0

In [6]:
df.isnull().sum()

Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64

In [7]:
# median for numeric columns
df["Quantity"] = df["Quantity"].fillna(df["Quantity"].median())
df["Price Per Unit"] = df["Price Per Unit"].fillna(df["Price Per Unit"].median())

# calculate missing total spent values by the equation: unit price * quantity
df["Total Spent"] = df["Total Spent"].fillna(df["Price Per Unit"] * df["Quantity"])

# if discount applied is not true so it should be false
df["Discount Applied"] = df["Discount Applied"].fillna(False)

# for Item column fill Nan with the most frequent Item (Mode Imputation)
most_frequent_item = df["Item"].mode()[0]
df["Item"] = df["Item"].fillna(most_frequent_item)

# for transaction date we create 3 columns for year, month and day
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"])
df["Year"] = df["Transaction Date"].dt.year
df["Month"] = df["Transaction Date"].dt.month
df["Day"] = df["Transaction Date"].dt.day

C:\Users\tahao\AppData\Local\Temp\ipykernel_13924\291551983.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Discount Applied"] = df["Discount Applied"].fillna(False)


In [8]:
df.isnull().sum()

Transaction ID      0
Customer ID         0
Category            0
Item                0
Price Per Unit      0
Quantity            0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Discount Applied    0
Year                0
Month               0
Day                 0
dtype: int64

In [9]:
# encode the location & payment method columns to numerical columns and drop the last columns in the alphabetical order
df_encoded = pd.get_dummies(df, columns= ["Location", "Payment Method"], drop_first = True)

# apply normalization for the columns: Price Per Unit, Quantity, Total Spent
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
numeric_columns = ["Price Per Unit", "Quantity", "Total Spent"]
df_encoded[numeric_columns] = scaler.fit_transform(df_encoded[numeric_columns])
df_encoded.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Transaction Date,Discount Applied,Year,Month,Day,Location_Online,Payment Method_Credit Card,Payment Method_Digital Wallet
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,0.375000,1.000000,0.444444,2024-04-08,True,2024,4,8,True,False,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,0.666667,0.888889,0.632099,2023-07-23,True,2023,7,23,True,False,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,0.458333,0.111111,0.093827,2022-10-05,False,2022,10,5,True,True,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,0.625000,0.888889,0.598765,2022-05-07,False,2022,5,7,True,True,False
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,0.208333,0.666667,0.203704,2022-10-02,False,2022,10,2,True,False,True


In [11]:
# apply k-means algorithm(k = 3)
from sklearn.cluster import KMeans
X_mining = df_encoded[['Price Per Unit', 'Quantity', 'Total Spent']]
kmeans = KMeans(n_clusters=3, random_state=42)
df['Cluster'] = kmeans.fit_predict(X_mining)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_mining, df['Cluster'], test_size=0.2, random_state=42)

# training decision tree model
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
dt_model = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)
dt_acc = accuracy_score(y_test, dt_pred)

In [12]:
print(f"Decision Tree Accuracy: {dt_acc * 100:.2f}%")
print("\n" + "="*50 + "\n")
# printing the tree conditions
from sklearn.tree import export_text
tree_rules = export_text(dt_model, feature_names=['Price Per Unit', 'Quantity', 'Total Spent'])
print("tree rules:")
print(tree_rules)

Decision Tree Accuracy: 95.83%


tree rules:
|--- Total Spent <= 0.36
|   |--- Quantity <= 0.39
|   |   |--- Price Per Unit <= 0.19
|   |   |   |--- class: 0
|   |   |--- Price Per Unit >  0.19
|   |   |   |--- class: 0
|   |--- Quantity >  0.39
|   |   |--- Price Per Unit <= 0.56
|   |   |   |--- class: 1
|   |   |--- Price Per Unit >  0.56
|   |   |   |--- class: 0
|--- Total Spent >  0.36
|   |--- Price Per Unit <= 0.44
|   |   |--- class: 1
|   |--- Price Per Unit >  0.44
|   |   |--- Price Per Unit <= 0.52
|   |   |   |--- class: 2
|   |   |--- Price Per Unit >  0.52
|   |   |   |--- class: 2

